In [8]:
!pip install llama-index-utils-workflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 68.8 MB/s eta 0:00:00


We can create a single-step workflow by defining a class that inherits from Workflow and decorating your functions with `@step`. We will also need to add `StartEvent` and `StopEvent`, which are special events that are used to indicate the start and end of the workflow.

In [3]:
from llama_index.core.workflow import StartEvent , StopEvent , Workflow , step

class MyWorkflow(Workflow):
    @step
    async def my_step(self , ev: StartEvent) -> StopEvent:
        print(f"1 + 2 -> {1+2}")
        return StopEvent(result = "Hello World")

w = MyWorkflow(timeout = 10 , verbose = False)
result = await w.run()
print(result)

1 + 2 -> 3
Hello World


### Connecting Multiple Steps
To connect multiple steps, we create custom events that carry data between steps. To do so, we need to add an `Event` that is passed between the steps and transfers the output of the first step to the second step.

In [4]:
from llama_index.core.workflow import Event

class ProcessingEvent(Event):
    intermediate_result: str

class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self , ev: StartEvent) -> ProcessingEvent:
        # Process initial data
        print("1 + 2 -> 3")
        return ProcessingEvent(intermediate_result = "Step 1 complete")

    @step
    async def step_two(self , ev: ProcessingEvent) -> StopEvent:
        # Use the intermediate result
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)

w = MultiStepWorkflow(timeout=10, verbose=False)
result = await w.run()
result

1 + 2 -> 3


'Finished processing: Step 1 complete'

### Loops and Branches
The type hinting is the most powerful part of workflows because it allows us to create branches, loops, and joins to facilitate more complex workflows.

Let’s show an example of creating a loop by using the union operator |. In the example below, we see that the LoopEvent is taken as input for the step and can also be returned as output.

In [20]:
from llama_index.core.workflow import Event
import random

class ProcessingEvent(Event):
    intermediate_result : str

class LoopEvent(Event):
    loop_output: str

class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> ProcessingEvent | LoopEvent:
        if random.randint(0, 2) == 0:
            print("Bad thing happened")
            return LoopEvent(loop_output="Back to step one.")
        else:
            print("Good thing happened")
            return ProcessingEvent(intermediate_result="First step complete.")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        # Use the intermediate result
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)

w = MultiStepWorkflow(verbose=False)
result = await w.run()
result

Bad thing happened
Bad thing happened
Good thing happened


'Finished processing: First step complete.'

### Drawing Workflows
We can also draw workflows. Let’s use the draw_all_possible_flows function to draw the workflow. This stores the workflow in an HTML file.

In [22]:
from llama_index.utils.workflow import draw_all_possible_flows

w = MultiStepWorkflow(verbose=False)
draw_all_possible_flows(w, "flow.html")

flow.html


### State Management
State management is useful when you want to keep track of the state of the workflow, so that every step has access to the same state. We can do this by using the `Context` type hint on top of a parameter in the step function.

In [24]:
from llama_index.core.workflow import Context , StartEvent , StopEvent , Workflow, step

class MyNewWorkflow(Workflow):
    @step
    async def query(self, ctx: Context, ev: StartEvent) -> StopEvent:
        # store query in the context
        await ctx.store.set("query", "What is the capital of France?")
        # do something with context and event
        val = 1 + 2

        # retrieve query from the context
        query_val = await ctx.store.get("query")

        return StopEvent(result=val)

### Automating workflows with Multi-Agent Workflows
Instead of manual workflow creation, we can use the `AgentWorkflow` class to create a multi-agent workflow. The `AgentWorkflow`: uses Workflow Agents to allow you to create a system of one or more agents that can collaborate and hand off tasks to each other based on their specialized capabilities. This enables building complex agent systems where different agents handle different aspects of a task. Instead of importing classes from `llama_index.core.agent`, we will import the agent classes from `llama_index.core.agent.workflow` . One agent must be designated as the root agent in the `AgentWorkflow` constructor. When a user message comes in, it is first routed to the root agent.

Each agent can then:
- Handle the request directly using their tools
- Handoff to another agent better suited for the task
- Return a response to the user


In [27]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
hf_token = os.environ["HF_TOKEN"]

In [31]:
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

async def main():
    # Instantiate HuggingFaceInferenceAPI directly within the async function
    llm = HuggingFaceInferenceAPI(
        model_name="Qwen/Qwen2.5-Coder-32B-Instruct",
        token=hf_token
    )

    # we can pass functions directly without FunctionTool -- the fn/docstring are parsed for the name/description
    multiply_agent = ReActAgent(
        name = "multiply_agent",
        description = "Is able to multiply two integers",
        system_prompt="A helpful assistant that can use a tool to multiply numbers.",
        tools=[multiply],
        llm=llm,
    )

    addition_agent = ReActAgent(
        name="add_agent",
        description="Is able to add two integers",
        system_prompt="A helpful assistant that can use a tool to add numbers.",
        tools=[add],
        llm=llm,
    )

    workflow = AgentWorkflow(
        agents=[multiply_agent, addition_agent],
        root_agent="multiply_agent",
    )

    # Run the system
    response = await workflow.run(user_msg="Can you add 5 and 3?")

    # Explicitly close the underlying client if it has an aclose method
    if hasattr(llm.client, 'aclose') and callable(llm.client.aclose):
        await llm.client.aclose()

    print(f"Workflow response: {response}")

# Call the async main function
await main()

RuntimeError: Cannot send a request, as the client has been closed.

In [32]:
from llama_index.core.workflow import Context

# Define some tools
async def add(ctx: Context, a: int, b: int) -> int:
    """Add two numbers."""
    # update our count
    cur_state = await ctx.store.get("state")
    cur_state["num_fn_calls"] += 1
    await ctx.store.set("state", cur_state)

    return a + b

async def multiply(ctx: Context, a: int, b: int) -> int:
    """Multiply two numbers."""
    # update our count
    cur_state = await ctx.store.get("state")
    cur_state["num_fn_calls"] += 1
    await ctx.store.set("state", cur_state)

    return a * b

...

workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent",
    initial_state={"num_fn_calls": 0},
    state_prompt="Current state: {state}. User message: {msg}",
)

# run the workflow with context
ctx = Context(workflow)
response = await workflow.run(user_msg="Can you add 5 and 3?", ctx=ctx)

# pull out and inspect the state
state = await ctx.store.get("state")
print(state["num_fn_calls"])

RuntimeError: Cannot send a request, as the client has been closed.